### Which are the top 10 pizza restaurants by rating?

In [1]:
import duckdb
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [2]:
conn=duckdb.connect('my_database.duckdb')

In [3]:
deliveroo_pizzeria=conn.execute(
    """ 
SELECT 
    resto.name AS name_resto, 
    resto.rating, resto.rating_number
FROM deliveroo_restaurants AS resto
JOIN deliveroo_categories AS cat
    ON resto.id = cat.restaurant_id
WHERE LOWER(resto.category) = 'italian'
  AND LOWER(cat.name) = 'pizza'
   AND CAST(REGEXP_REPLACE(resto.rating_number, '[^0-9]', '', 'g') AS INTEGER) > 10
GROUP BY 
    resto.name,resto.rating,resto.rating_number
ORDER BY resto.rating DESC
LIMIT 10;


"""
).fetch_df()
deliveroo_pizzeria

,name_resto,rating,rating_number
0,Orso Pizzeria,4.4,457
1,Pasta Commedia,4.4,129
2,Pizarro Antwerpen,4.4,500+
3,Napoli Da Leonardo,4.4,78
4,Pasta Presto,4.3,223
5,Gran Gusto Ristorante,4.3,26
6,Benvenuto,4.3,50
7,La Trattoria Italiana,4.2,306
8,Giussepe's,4.2,500+
9,Bella Italia,4.2,163


In [4]:
takeaway_pizzeria=conn.execute(
    """ 
SELECT resto.name AS restaurant,
       resto.ratings AS rating, 
       resto.ratingsNumber AS rating_number
FROM takeaway_restaurants AS resto
JOIN takeaway_categories AS cat
    ON resto.primarySlug = cat.restaurant_id
JOIN takeaway_menuItems AS menu
    ON resto.primarySlug = menu.primarySlug
WHERE LOWER(cat.name) LIKE '%ital%' 
  AND LOWER(menu.name) LIKE '%izza%' 
  AND LOWER(resto.name) NOT LIKE '%keba%'

AND CAST(REGEXP_REPLACE(resto.ratingsNumber, '[^0-9]', '', 'g') AS INTEGER) > 100
GROUP BY resto.name, resto.ratings, resto.ratingsNumber
ORDER BY resto.ratings DESC
LIMIT 10;


"""
).fetch_df()
takeaway_pizzeria

,restaurant,rating,rating_number
0,Asya Pizzeria,4.7,1151
1,Ali Baba,4.6,673
2,Pizzeria Per Passione,4.6,1799
3,Pizza Istanbul,4.6,777
4,BE-JACKS,4.5,5674
5,Efes Geetbets,4.5,182
6,Orient KURINGEN,4.4,1059
7,Dolce Vita,4.4,418
8,Pizzeria Valsugana,4.4,555
9,Aladin,4.3,798


In [ ]:
takeaway_pizzeria_sanscond=conn.execute(
    """ 
SELECT resto.name AS restaurant,
       resto.ratings AS rating, 
       resto.ratingsNumber AS rating_number
FROM takeaway_restaurants AS resto
JOIN takeaway_categories AS cat
    ON resto.primarySlug = cat.restaurant_id
WHERE LOWER(cat.name) LIKE '%pizza%'  
  AND LOWER(resto.name) NOT LIKE '%kebab%' AND LOWER(resto.name) NOT LIKE '%kebap%'
  AND CAST(resto.ratingsNumber AS INTEGER)> 100 
  AND LOWER(resto.name) NOT LIKE '%domino''s pizza%' 
  AND LOWER(resto.name) NOT LIKE '%hut%'  -- Escludi Pizza Hut Pizza 
  AND LOWER(resto.name) NOT LIKE '%heat%'  -- Escludi Pizza Hut 
  AND LOWER(resto.name) NOT LIKE '%heat%'  
  AND LOWER(resto.name) NOT LIKE '%cut%' 
  AND LOWER(resto.name) NOT LIKE '%minute%' 
    AND LOWER(resto.name) NOT LIKE '%service%' 
GROUP BY resto.name, resto.ratings, resto.ratingsNumber
ORDER BY resto.ratings DESC
LIMIT 10;

"""
).fetch_df()
takeaway_pizzeria_sanscond

,restaurant,rating,rating_number
0,De Echte Eethuis Carlos,5,599
1,Pizzeria Zirar Saint-Denis,4.9,529
2,Baskent Meerhout,4.9,879
3,Am Princesse,4.9,367
4,Pitta de Kroon,4.9,261
5,More pizza's,4.9,311
6,Pizza Time Evergem,4.9,1148
7,De Notenboom,4.9,706
8,The Black Horse,4.9,127
9,Pizza L'Esta,4.9,372


In [6]:
ubereats_pizzeria = conn.execute(
    """
    SELECT resto.title, resto.rating__rating_value, resto.rating__review_count
    FROM ubereats_restaurants AS resto
    JOIN ubereats_restaurant_to_categories AS resto_cat
        ON resto.id = resto_cat.restaurant_id
    WHERE (LOWER(resto_cat.category) LIKE '%ital%' 
        OR LOWER(resto_cat.category) LIKE '%napol%' 
        OR LOWER(resto_cat.category) LIKE '%pizza%')
      AND CAST(REGEXP_REPLACE(resto.rating__review_count, '[^0-9]', '', 'g') AS INTEGER) > 100
      AND LOWER(resto.title) NOT LIKE '%domino''s pizza%' 
      AND LOWER(resto.title) NOT LIKE '%hut%'  -- Escludi Pizza Hut
    GROUP BY resto.title, resto.rating__rating_value, resto.rating__review_count
    ORDER BY resto.rating__rating_value DESC
    LIMIT 10;
    """
).fetch_df()
ubereats_pizzeria

,title,rating__rating_value,rating__review_count
0,MiTo Orban / Milano Torino,4.7,200+
1,Woodiz Basilique,4.6,148
2,La Cucina - Pizzeria Tradizionale,4.5,199
3,Pizzeria Per Passione,4.5,124
4,Chez Giovanni,4.5,200+
5,La Cucina - Pizzeria Tradizionale - Anderlecht,4.5,178
6,Dawag,4.4,133
7,La Trinacria,4.3,160
8,Pizzeria Romeo e Giulietta,4.3,129
9,Arrivero Pizza - Schaerbeek,4.2,116


In [7]:
conn.close()